### 오디오 녹음에서 화자 구분하기

In [19]:
#!pip install --upgrade google-cloud-speech
#!pip install vertexai

In [ ]:
!gcloud auth application-default login

### Local에 저장된 짧은 음성 파일 인식

In [11]:
PROJECT_ID = "sesac-dev-400904"
GCS_URI = "resources/commercial_mono.wav"

In [12]:
from google.cloud.speech_v2 import SpeechClient
from google.cloud.speech_v2.types import cloud_speech

# Instantiates a client
client = SpeechClient()

# Reads a file as bytes
with open(GCS_URI, "rb") as f:
    audio_content = f.read()

config = cloud_speech.RecognitionConfig(
    auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
    language_codes=["en-US"],
    model="long",
)

request = cloud_speech.RecognizeRequest(
    recognizer=f"projects/{PROJECT_ID}/locations/global/recognizers/_",
    config=config,
    content=audio_content,
)

# Transcribes the audio into text
response = client.recognize(request=request)

for result in response.results:
    print(f"Transcript: {result.alternatives[0].transcript}")

Transcript: okay I'm here
Transcript:  hi I'd like to buy a Chromecast and I was wondering whether you could help me with that
Transcript:  certainly which color would you like we have blue black and red
Transcript:  let's get the black one
Transcript:  okay great would you like the new Chromecast Ultra model or the regular Chromecast
Transcript:  regular Chromecast is fine
Transcript:  okay sure would you like to ship it regular or Express
Transcript:  Express please
Transcript:  terrific it's on the way thank you very much thank you
Transcript:  bye


In [ ]:
# 필요한 라이브러리 설치
# pip install google-cloud-aiplatform

import vertexai
from vertexai.generative_models import GenerativeModel, Part


def transcribe_diarize_gemini(
    project_id: str, location: str, gcs_audio_uri: str
) -> str:
    """
    Gemini 1.5 Pro 모델을 사용하여 GCS의 오디오 파일을 변환하고 화자를 분리합니다.
    """
    # Vertex AI 초기화
    vertexai.init(project=project_id, location=location)

    # Gemini 1.5 Pro 모델 로드
    model = GenerativeModel("gemini-2.5-pro")

    # 오디오 파일과 텍스트 프롬프트 준비
    audio_file = Part.from_uri(
        mime_type="audio/wav", uri=gcs_audio_uri  # 오디오 파일 형식에 맞게 변경
    )

    prompt = """
    당신은 전문적인 회의록 작성자입니다. 제공된 오디오 파일을 듣고 다음 작업을 수행해 주십시오:
    1. 전체 대화를 정확하게 텍스트로 변환합니다.
    2. 각 발화에 대해 화자를 구분합니다. 화자 이름이 언급되면 해당 이름을 사용하고, 그렇지 않으면 "화자 A", "화자 B" 등으로 구분합니다.
    3. 최종 결과는 아래의 JSON 형식과 정확히 일치해야 합니다. 각 JSON 객체는 'speaker', 'start_time_seconds', 'transcript' 키를 포함해야 합니다.
    """

    # 모델에 멀티모달 프롬프트 전송
    response = model.generate_content([audio_file, prompt])

    # 결과 출력
    # Gemini 응답은 Markdown 코드 블록을 포함할 수 있으므로, 이를 정리합니다.
    cleaned_response = response.text.strip().replace("```json", "").replace("```", "")
    print(cleaned_response)

    return cleaned_response


# 사용 예시
PROJECT_ID = "sesac-dev-400904"
LOCATION = "us-central1"
GCS_URI = "gs://sesac-dev-400904-my-new-bucket/uploads/commercial_mono.wav"
transcribe_diarize_gemini(PROJECT_ID, LOCATION, GCS_URI)


[
  {
    "speaker": "화자 A",
    "start_time_seconds": 1.25,
    "transcript": "Okay, I'm here."
  },
  {
    "speaker": "화자 B",
    "start_time_seconds": 4.14,
    "transcript": "Hi, I'd like to buy a Chromecast and I was wondering whether you could help me with that."
  },
  {
    "speaker": "화자 A",
    "start_time_seconds": 9.47,
    "transcript": "Uh certainly, which color would you like? We have blue, black, and red."
  },
  {
    "speaker": "화자 B",
    "start_time_seconds": 14.16,
    "transcript": "Let's get the black one."
  },
  {
    "speaker": "화자 A",
    "start_time_seconds": 17.2,
    "transcript": "Uh okay, great. Would you like the new Chromecast Ultra model or the regular Chromecast?"
  },
  {
    "speaker": "화자 B",
    "start_time_seconds": 22.3,
    "transcript": "Mm, regular Chromecast is fine."
  },
  {
    "speaker": "화자 A",
    "start_time_seconds": 25.32,
    "transcript": "Okay, sure. Would you like to ship it regular or express?"
  },
  {
    "speaker": "화자 B"

'\n[\n  {\n    "speaker": "화자 A",\n    "start_time_seconds": 1.25,\n    "transcript": "Okay, I\'m here."\n  },\n  {\n    "speaker": "화자 B",\n    "start_time_seconds": 4.14,\n    "transcript": "Hi, I\'d like to buy a Chromecast and I was wondering whether you could help me with that."\n  },\n  {\n    "speaker": "화자 A",\n    "start_time_seconds": 9.47,\n    "transcript": "Uh certainly, which color would you like? We have blue, black, and red."\n  },\n  {\n    "speaker": "화자 B",\n    "start_time_seconds": 14.16,\n    "transcript": "Let\'s get the black one."\n  },\n  {\n    "speaker": "화자 A",\n    "start_time_seconds": 17.2,\n    "transcript": "Uh okay, great. Would you like the new Chromecast Ultra model or the regular Chromecast?"\n  },\n  {\n    "speaker": "화자 B",\n    "start_time_seconds": 22.3,\n    "transcript": "Mm, regular Chromecast is fine."\n  },\n  {\n    "speaker": "화자 A",\n    "start_time_seconds": 25.32,\n    "transcript": "Okay, sure. Would you like to ship it regular or e